**MENTR analysis on the ICM cluster - pull outputs**    
B Rioux
22 April 2026

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# ── Set up figure ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5),
                         gridspec_kw={'width_ratios': [1.1, 1]})
fig.suptitle('Balanced (Horizontal) Pleiotropy', fontsize=16, fontweight='bold', y=1.01)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Panel A – MR Scatter Plot with Balanced Pleiotropy
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ax = axes[0]
np.random.seed(42)

n_snps = 25
true_causal = 0.50

# SNP–exposure effects (all positive – valid instruments)
beta_x = np.random.uniform(0.04, 0.40, n_snps)

# Balanced pleiotropic effects: mean ≈ 0
pleiotropy = np.random.normal(0, 0.04, n_snps)

# SNP–outcome = causal effect × SNP–exposure + balanced pleiotropy + noise
beta_y = true_causal * beta_x + pleiotropy + np.random.normal(0, 0.015, n_snps)

se_x = np.random.uniform(0.010, 0.030, n_snps)
se_y = np.random.uniform(0.020, 0.045, n_snps)

# Points with error bars
ax.errorbar(beta_x, beta_y, xerr=se_x, yerr=se_y,
            fmt='o', color='#4C72B0', alpha=0.70, markersize=7,
            capsize=2, elinewidth=0.8, label='SNP instruments')

# IVW line (through origin)
x_line = np.linspace(0, 0.45, 200)
ax.plot(x_line, true_causal * x_line, '-',
        color='#C44E52', lw=2.2, label=f'IVW  (β = {true_causal:.2f})')

# MR-Egger line (with intercept — should be ≈ 0 under balanced pleiotropy)
egger_coefs = np.polyfit(beta_x, beta_y, 1)        # [slope, intercept]
ax.plot(x_line, np.polyval(egger_coefs, x_line), '--',
        color='#55A868', lw=2.2,
        label=f'MR-Egger  (int = {egger_coefs[1]:+.3f})')

ax.axhline(0, color='grey', ls=':', lw=0.7, alpha=0.6)
ax.axvline(0, color='grey', ls=':', lw=0.7, alpha=0.6)
ax.set_xlabel('SNP – Exposure association  ($\\hat\\beta_{Xj}$)', fontsize=11)
ax.set_ylabel('SNP – Outcome association  ($\\hat\\beta_{Yj}$)', fontsize=11)
ax.set_title('A.  MR scatter plot', fontsize=13, fontweight='bold', loc='left')
ax.legend(fontsize=9, framealpha=0.9)

# Annotation
ax.annotate('Pleiotropic effects scatter\nsymmetrically around the\ncausal-estimate line\n→ intercept ≈ 0',
            xy=(0.30, true_causal*0.30 + 0.04), fontsize=8.5,
            xytext=(0.08, 0.20),
            arrowprops=dict(arrowstyle='->', color='#555555', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.35', fc='#FFFDE7', ec='#CCCCCC'),
            color='#333333')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Panel B – Conceptual DAG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ax2 = axes[1]
ax2.set_xlim(-0.5, 10.5)
ax2.set_ylim(-0.5, 10.5)
ax2.axis('off')
ax2.set_title('B.  Conceptual diagram', fontsize=13, fontweight='bold', loc='left')

# ── helper: draw a rounded box ────────────────────────────────
def draw_box(ax, xy, text, color='#E3F2FD', ec='#1565C0', fontsize=11):
    w, h = 2.2, 1.0
    box = FancyBboxPatch((xy[0] - w/2, xy[1] - h/2), w, h,
                         boxstyle="round,pad=0.15", fc=color, ec=ec, lw=1.6)
    ax.add_patch(box)
    ax.text(xy[0], xy[1], text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=ec)

# ── nodes ─────────────────────────────────────────────────────
draw_box(ax2, (1.5, 5), 'G\n(SNP)', color='#FFF9C4', ec='#F57F17')
draw_box(ax2, (5, 8.5), 'Exposure\n(X)', color='#E3F2FD', ec='#1565C0')
draw_box(ax2, (9, 5), 'Outcome\n(Y)', color='#FFEBEE', ec='#C62828')
draw_box(ax2, (5, 5), 'Pathway 1', color='#E8F5E9', ec='#2E7D32', fontsize=9.5)
draw_box(ax2, (5, 1.5), 'Pathway 2', color='#F3E5F5', ec='#6A1B9A', fontsize=9.5)

# ── arrows ────────────────────────────────────────────────────
arrow_kw = dict(arrowstyle='->', lw=2, mutation_scale=18)

# G → X  (causal pathway — thick)
ax2.annotate('', xy=(3.8, 8.5), xytext=(2.7, 5.6),
             arrowprops=dict(**arrow_kw, color='#1565C0'))
ax2.text(2.6, 7.4, '$\\beta_{GX}$', fontsize=10, color='#1565C0', fontweight='bold')

# X → Y  (causal effect)
ax2.annotate('', xy=(7.8, 5.7), xytext=(6.2, 8.2),
             arrowprops=dict(**arrow_kw, color='#C62828'))
ax2.text(7.4, 7.4, '$\\beta_{XY}$', fontsize=10, color='#C62828', fontweight='bold')

# G → Pathway 1 (positive pleiotropy)
ax2.annotate('', xy=(3.8, 5), xytext=(2.7, 5),
             arrowprops=dict(**arrow_kw, color='#2E7D32'))

# Pathway 1 → Y  (+δ)
ax2.annotate('', xy=(7.8, 5.2), xytext=(6.2, 5.1),
             arrowprops=dict(**arrow_kw, color='#2E7D32'))
ax2.text(6.8, 5.55, '$+\\delta_1$', fontsize=10, color='#2E7D32', fontweight='bold')

# G → Pathway 2 (negative pleiotropy)
ax2.annotate('', xy=(3.8, 1.5), xytext=(2.7, 4.4),
             arrowprops=dict(**arrow_kw, color='#6A1B9A'))

# Pathway 2 → Y  (−δ)
ax2.annotate('', xy=(7.8, 4.5), xytext=(6.2, 2.0),
             arrowprops=dict(**arrow_kw, color='#6A1B9A'))
ax2.text(6.8, 2.8, '$-\\delta_2$', fontsize=10, color='#6A1B9A', fontweight='bold')

# ── balance annotation ────────────────────────────────────────
ax2.text(5, -0.2,
         'Balanced pleiotropy:  $\\mathbb{E}[\\delta] = 0$\n'
         '($+\\delta_1$ and $-\\delta_2$ cancel on average)',
         ha='center', va='top', fontsize=10,
         bbox=dict(boxstyle='round,pad=0.4', fc='#FFFDE7', ec='#BDBDBD'),
         color='#333333')

plt.tight_layout()
plt.savefig('balanced_pleiotropy.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

: 

Select R Kernel in Jupyter kernels -> R (r_env)

In [ ]:
library(dplyr)
library(data.table)
library(R.utils)
library(openxlsx)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘pac

# Load ontology mapping

In [ ]:
# Load ontology file (sample_ontology_ID -> cell type term)
ontology <- fread("/network/iss/debette/users/bastien.rioux/mentr/MENTR/resources/supp_table_10.sample_ontology_information.tsv.gz")

# Check it
head(ontology[, .(sample_ontology_ID, sample_ontology_term, sample_ontology_type)])

# Load CAGE cluster info (clusterID -> gene name, type, etc.)
cage_info <- fread("/network/iss/debette/users/bastien.rioux/mentr/MENTR/resources/F5.cage_cluster.hg19.info.tsv.gz")

# Check it
head(cage_info[, .(clusterID, clusterName, type, geneNameStr, geneClassStr)])

sample_ontology_ID,sample_ontology_term,sample_ontology_type
<chr>,<chr>,<chr>
CL:0000034,stem cell,cell_type
CL:0000037,hematopoietic stem cell,cell_type
CL:0000047,neuronal stem cell,cell_type
CL:0000049,common myeloid progenitor,cell_type
CL:0000056,myoblast,cell_type
CL:0000057,fibroblast,cell_type


clusterID,clusterName,type,geneNameStr,geneClassStr
<chr>,<chr>,<chr>,<chr>,<chr>
"chr10:100013403..100013414,-",p2@LOXL4,promoter,LOXL4,coding_mRNA
chr10:100019738-100019959,e1@ADDG10100019738.E,enhancer,ADDG10100019738.E,enhancer_locus
chr10:100020230-100020246,e1@ADDG10100020230.E,enhancer,ADDG10100020230.E,enhancer_locus
"chr10:100027943..100027958,-",p1@LOXL4,promoter,LOXL4,coding_mRNA
chr10:100074404-100074582,e1@ADDG10100074404.E,enhancer,ADDG10100074404.E,enhancer_locus
chr10:100076038-100076149,e1@ADDG10100076038.E,enhancer,ADDG10100076038.E,enhancer_locus


# (Explore test run)

Read the output file named all_qcd_res.txt.gz located in the directory /network/iss/debette/users/bastien.rioux/mentr/output_test/output/ using the read.csv function in R. The file is tab-separated and contains a header row.

In [ ]:
all_qcd_res <- fread("/network/iss/debette/users/bastien.rioux/mentr/output_test/output/all_qcd_res.txt.gz")

In [ ]:
str(all_qcd_res)

Classes ‘data.table’ and 'data.frame':	235 obs. of  356 variables:
 $ CHR           : chr  "chr1" "chr1" "chr1" "chr1" ...
 $ POS           : int  1999288 2001856 2004947 2008444 2001273 1999288 2001856 2004947 2008444 2001273 ...
 $ REF           : chr  "T" "A" "T" "T" ...
 $ ALT           : chr  "C" "G" "C" "C" ...
 $ dist          : int  89045 91613 94704 98201 91030 84577 87145 90236 93733 86562 ...
 $ gene          : chr  "chr1:1910241..1910262,+" "chr1:1910241..1910262,+" "chr1:1910241..1910262,+" "chr1:1910241..1910262,+" ...
 $ strand        : chr  "+" "+" "+" "+" ...
 $ ref_matched   : logi  TRUE TRUE TRUE TRUE TRUE TRUE ...
 $ CL:0000034    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000037    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000047    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000049    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000056    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000057    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000062    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000066    : num

In [ ]:
# Keep only reliable records
dim(all_qcd_res)
all_res_qced <- all_qcd_res[ref_matched == TRUE]
dim(all_res_qced)

[1] 235 356

[1] 235 356

> Read .csv file (hg19) containing rsid and chr/pos/ea/nea (needs to be downloaded directly from DNA Nexus with wget in mentr/input/)

DNA nexus path: /shivai_analyses/gwas/postgwas/mentr/input/

In CLI:
- cd /network/iss/debette/users/bastien.rioux/mentr/input/
- wget https://dl.ew2.dnanex.us/F/D/J20j9J68bJ4XPXG2x9vfXB1VfP9JXyv3YfXJjYv8/mentr_input_hg19.csv

In [ ]:
rsid_table <- fread("/network/iss/debette/users/bastien.rioux/mentr/input/mentr_input_hg19.csv")
head(rsid_table)

rsid,chr,pos,ea,nea
<chr>,<chr>,<int>,<chr>,<chr>
rs2459985,1,1999288,C,T
rs2803303,1,2001856,G,A
rs2678942,1,2004947,C,T
rs13302948,1,2008444,C,T
rs2678941,1,2001273,G,A
rs2254669,1,2006108,G,A


> CS file to map rsID

DNA nexus path: /shivai_analyses/gwas/postgwas/susie_r/

in CLI:
- cd /network/iss/debette/users/bastien.rioux/mentr/input/ 
- wget https://dl.ew2.dnanex.us/F/D/JZvVvjQqj309jZjzFKzKK7YY4y4kby69Y21fz81X/susierss_cs_snps_20260417.xlsx

In [ ]:
cs_table <- read.xlsx("/network/iss/debette/users/bastien.rioux/mentr/input/susierss_cs_snps_20260417.xlsx")
# Add susierss prefix
rsid_col <- "rsID"   # ← change if column is named differently e.g. "rsid"
colnames(cs_table) <- ifelse(
  colnames(cs_table) == rsid_col,
  colnames(cs_table),                            # keep rsID as-is
  paste0("susierss_", colnames(cs_table))        # prefix all others
)
cat("\ncs_table columns after prefix:\n")
print(colnames(cs_table))


cs_table columns after prefix:
 [1] "susierss_trait"              "susierss_risk_loci_num"     
 [3] "susierss_cs"                 "rsID"                       
 [5] "susierss_pip"                "susierss_pip_rank"          
 [7] "susierss_locus_gene"         "susierss_locus_start"       
 [9] "susierss_locus_end"          "susierss_locus_cytoband"    
[11] "susierss_CHR"                "susierss_POS"               
[13] "susierss_EA"                 "susierss_NEA"               
[15] "susierss_EAF"                "susierss_BETA"              
[17] "susierss_SE"                 "susierss_P"                 
[19] "susierss_INFO"               "susierss_is_indep_snp"      
[21] "susierss_indep_snp_method"   "susierss_snp_gene"          
[23] "susierss_snp_func"           "susierss_NOVEL"             
[25] "susierss_b2_replicated"      "susierss_b2_replicated_bonf"


> Format output

In [ ]:
library(data.table)

# ── Harmonize CHR format ──────────────────────────────────────────────────────
rsid_table[, chr := paste0("chr", chr)]

cat("df_long CHR examples:    ", head(df_long$CHR), "\n")
cat("rsid_table chr examples: ", head(rsid_table$chr), "\n")

# ── Fixed columns (metadata) ──────────────────────────────────────────────────
meta_cols <- c("CHR", "POS", "REF", "ALT", "dist", "gene", "strand", "ref_matched")

# ── Cell type columns ─────────────────────────────────────────────────────────
cell_cols <- setdiff(colnames(all_res_qced), meta_cols)

# ── Melt to long format ───────────────────────────────────────────────────────
df_long <- melt(all_res_qced,
                id.vars       = meta_cols,
                variable.name = "cell_type",
                value.name    = "delta_prob")

# ── Join rsID ─────────────────────────────────────────────────────────────────
df_long <- merge(
  df_long,
  rsid_table[, .(rsid, chr, pos, ea, nea)],
  by.x  = c("CHR", "POS", "ALT", "REF"),
  by.y  = c("chr", "pos", "ea",  "nea"),
  all.x = TRUE
)

n_total   <- nrow(df_long)
n_matched <- df_long[!is.na(rsid), .N]
cat("Variants with rsID   :", n_matched, "\n")
cat("Variants without rsID:", n_total - n_matched, "\n")

# ── Join ontology ─────────────────────────────────────────────────────────────
df_long <- merge(
  df_long,
  ontology[, .(sample_ontology_ID,
               sample_ontology_term,
               sample_ontology_type)],
  by.x  = "cell_type",
  by.y  = "sample_ontology_ID",
  all.x = TRUE
)

# ── Join CAGE info ────────────────────────────────────────────────────────────
df_long <- merge(
  df_long,
  cage_info[, .(clusterID,
                clusterName,
                type,
                geneNameStr,
                geneClassStr)],
  by.x  = "gene",
  by.y  = "clusterID",
  all.x = TRUE
)

# ── Join SuSiE RSS CS table by rsid ──────────────────────────────────────────
df_long <- merge(
  df_long,
  cs_table,             # all susierss_ prefixed cols + rsID
  by.x  = "rsid",
  by.y  = "rsID",       # ← original rsID col name before prefix
  all.x = TRUE          # keep all df_long rows
)

# Check match rate
n_in_cs <- df_long[!is.na(susierss_cs), .N]   # ← replace susierss_cs with any known col
cat("Variants matched to SuSiE CS table:", n_in_cs, "\n")

# ── Reorder columns ───────────────────────────────────────────────────────────
# Get all susierss_ columns dynamically
susierss_cols <- grep("^susierss_", colnames(df_long), value = TRUE)

col_order <- c("gene", "clusterName", "type", "geneNameStr", "geneClassStr",
               "cell_type", "sample_ontology_term", "sample_ontology_type",
               "rsid", "CHR", "POS", "REF", "ALT", "dist", "strand", "delta_prob",
               susierss_cols)   # ← appended at the end

df_long <- df_long[, ..col_order]

# Preview
head(df_long)
cat("\nFinal columns:\n")
print(colnames(df_long))

# ── Filter effects ────────────────────────────────────────────────────────────
robust <- df_long[abs(delta_prob) >= 0.1]
cat("Robust effects     (|Δprob| >= 0.1) :", nrow(robust), "\n")

permissive <- df_long[abs(delta_prob) >= 0.05]
cat("Permissive effects (|Δprob| >= 0.05):", nrow(permissive), "\n")

ERROR: Error: object 'df_long' not found


In [ ]:
head(permissive)

gene,clusterName,type,geneNameStr,geneClassStr,cell_type,sample_ontology_term,sample_ontology_type,rsid,CHR,⋯,susierss_SE,susierss_P,susierss_INFO,susierss_is_indep_snp,susierss_indep_snp_method,susierss_snp_gene,susierss_snp_func,susierss_NOVEL,susierss_b2_replicated,susierss_b2_replicated_bonf
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
"chr1:2005462..2005537,+",p7@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0000353,parenchyma,tissue,rs13302948,chr1,⋯,0.006090,4.17714e-08,0.997427,FALSE,NA,NA,NA,,,
"chr1:2005046..2005122,+",p1@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0001896,medulla oblongata,tissue,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,
"chr1:2005046..2005122,+",p1@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0004875,nephrogenic cord,tissue,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,
"chr1:2005126..2005153,+",p5@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0004151,cardiac chamber,tissue,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,
"chr1:2005182..2005194,+",p8@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0003513,trunk blood vessel,tissue,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,
"chr1:2005228..2005248,+",p3@PRKCZ,promoter,PRKCZ,coding_mRNA,CL:0000134,mesenchymal stem cell,cell_type,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,


In [ ]:
write.xlsx(permissive, 
           "/network/iss/debette/users/bastien.rioux/mentr/output_test/permissive_test.xlsx")

# (Explore parallel test run)

Read the output file named all_qcd_res.txt.gz located in the directory /network/iss/debette/users/bastien.rioux/mentr/output_test/output/ using the read.csv function in R. The file is tab-separated and contains a header row.

In [ ]:
all_qcd_res <- fread("/network/iss/debette/users/bastien.rioux/mentr/output_test_parallel/output/all_qcd_res.txt.gz")

In [ ]:
str(all_qcd_res)

Classes ‘data.table’ and 'data.frame':	235 obs. of  356 variables:
 $ CHR           : chr  "chr1" "chr1" "chr1" "chr1" ...
 $ POS           : int  1999288 2001856 2004947 2008444 2001273 1999288 2001856 2004947 2008444 2001273 ...
 $ REF           : chr  "T" "A" "T" "T" ...
 $ ALT           : chr  "C" "G" "C" "C" ...
 $ dist          : int  89045 91613 94704 98201 91030 84577 87145 90236 93733 86562 ...
 $ gene          : chr  "chr1:1910241..1910262,+" "chr1:1910241..1910262,+" "chr1:1910241..1910262,+" "chr1:1910241..1910262,+" ...
 $ strand        : chr  "+" "+" "+" "+" ...
 $ ref_matched   : logi  TRUE TRUE TRUE TRUE TRUE TRUE ...
 $ CL:0000034    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000037    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000047    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000049    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000056    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000057    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000062    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ CL:0000066    : num

In [ ]:
# Keep only reliable records
dim(all_qcd_res)
all_res_qced <- all_qcd_res[ref_matched == TRUE]
dim(all_res_qced)

[1] 235 356

[1] 235 356

> Read .csv file (hg19) containing rsid and chr/pos/ea/nea (needs to be downloaded directly from DNA Nexus with wget in mentr/input/)

DNA nexus path: /shivai_analyses/gwas/postgwas/mentr/input/

In CLI:
- cd /network/iss/debette/users/bastien.rioux/mentr/input/
- wget https://dl.ew2.dnanex.us/F/D/J20j9J68bJ4XPXG2x9vfXB1VfP9JXyv3YfXJjYv8/mentr_input_hg19.csv

In [ ]:
rsid_table <- fread("/network/iss/debette/users/bastien.rioux/mentr/input/mentr_input_hg19.csv")

# ── Harmonize CHR format ──────────────────────────────────────────────────────
rsid_table[, chr := paste0("chr", chr)]

cat("rsid_table chr examples: ", head(rsid_table$chr), "\n")
head(rsid_table)


rsid_table chr examples:  chr1 chr1 chr1 chr1 chr1 chr1 


rsid,chr,pos,ea,nea
<chr>,<chr>,<int>,<chr>,<chr>
rs2459985,chr1,1999288,C,T
rs2803303,chr1,2001856,G,A
rs2678942,chr1,2004947,C,T
rs13302948,chr1,2008444,C,T
rs2678941,chr1,2001273,G,A
rs2254669,chr1,2006108,G,A


> CS file to map rsID

DNA nexus path: /shivai_analyses/gwas/postgwas/susie_r/

in CLI:
- cd /network/iss/debette/users/bastien.rioux/mentr/input/ 
- wget https://dl.ew2.dnanex.us/F/D/JZvVvjQqj309jZjzFKzKK7YY4y4kby69Y21fz81X/susierss_cs_snps_20260417.xlsx

In [ ]:
cs_table <- read.xlsx("/network/iss/debette/users/bastien.rioux/mentr/input/susierss_cs_snps_20260417.xlsx")
# Add susierss prefix
rsid_col <- "rsID"   # ← change if column is named differently e.g. "rsid"
colnames(cs_table) <- ifelse(
  colnames(cs_table) == rsid_col,
  colnames(cs_table),                            # keep rsID as-is
  paste0("susierss_", colnames(cs_table))        # prefix all others
)
cat("\ncs_table columns after prefix:\n")
print(colnames(cs_table))


cs_table columns after prefix:
 [1] "susierss_trait"              "susierss_risk_loci_num"     
 [3] "susierss_cs"                 "rsID"                       
 [5] "susierss_pip"                "susierss_pip_rank"          
 [7] "susierss_locus_gene"         "susierss_locus_start"       
 [9] "susierss_locus_end"          "susierss_locus_cytoband"    
[11] "susierss_CHR"                "susierss_POS"               
[13] "susierss_EA"                 "susierss_NEA"               
[15] "susierss_EAF"                "susierss_BETA"              
[17] "susierss_SE"                 "susierss_P"                 
[19] "susierss_INFO"               "susierss_is_indep_snp"      
[21] "susierss_indep_snp_method"   "susierss_snp_gene"          
[23] "susierss_snp_func"           "susierss_NOVEL"             
[25] "susierss_b2_replicated"      "susierss_b2_replicated_bonf"


> Format output

In [ ]:
library(data.table)

# ── Fixed columns (metadata) ──────────────────────────────────────────────────
meta_cols <- c("CHR", "POS", "REF", "ALT", "dist", "gene", "strand", "ref_matched")

# ── Cell type columns ─────────────────────────────────────────────────────────
cell_cols <- setdiff(colnames(all_res_qced), meta_cols)

# ── Melt to long format ───────────────────────────────────────────────────────
df_long <- melt(all_res_qced,
                id.vars       = meta_cols,
                variable.name = "cell_type",
                value.name    = "delta_prob")

# ── Join rsID ─────────────────────────────────────────────────────────────────
df_long <- merge(
  df_long,
  rsid_table[, .(rsid, chr, pos, ea, nea)],
  by.x  = c("CHR", "POS", "ALT", "REF"),
  by.y  = c("chr", "pos", "ea",  "nea"),
  all.x = TRUE
)

n_total   <- nrow(df_long)
n_matched <- df_long[!is.na(rsid), .N]
cat("Variants with rsID   :", n_matched, "\n")
cat("Variants without rsID:", n_total - n_matched, "\n")

# ── Join ontology ─────────────────────────────────────────────────────────────
df_long <- merge(
  df_long,
  ontology[, .(sample_ontology_ID,
               sample_ontology_term,
               sample_ontology_type)],
  by.x  = "cell_type",
  by.y  = "sample_ontology_ID",
  all.x = TRUE
)

# ── Join CAGE info ────────────────────────────────────────────────────────────
df_long <- merge(
  df_long,
  cage_info[, .(clusterID,
                clusterName,
                type,
                geneNameStr,
                geneClassStr)],
  by.x  = "gene",
  by.y  = "clusterID",
  all.x = TRUE
)

# ── Join SuSiE RSS CS table by rsid ──────────────────────────────────────────
df_long <- merge(
  df_long,
  cs_table,             # all susierss_ prefixed cols + rsID
  by.x  = "rsid",
  by.y  = "rsID",       # ← original rsID col name before prefix
  all.x = TRUE          # keep all df_long rows
)

# Check match rate
n_in_cs <- df_long[!is.na(susierss_cs), .N]   # ← replace susierss_cs with any known col
cat("Variants matched to SuSiE CS table:", n_in_cs, "\n")

# ── Reorder columns ───────────────────────────────────────────────────────────
# Get all susierss_ columns dynamically
susierss_cols <- grep("^susierss_", colnames(df_long), value = TRUE)

col_order <- c("gene", "clusterName", "type", "geneNameStr", "geneClassStr",
               "cell_type", "sample_ontology_term", "sample_ontology_type",
               "rsid", "CHR", "POS", "REF", "ALT", "dist", "strand", "delta_prob",
               susierss_cols)   # ← appended at the end

df_long <- df_long[, ..col_order]

# Preview
head(df_long)
cat("\nFinal columns:\n")
print(colnames(df_long))

# ── Filter effects ────────────────────────────────────────────────────────────
robust <- df_long[abs(delta_prob) >= 0.1]
cat("Robust effects     (|Δprob| >= 0.1) :", nrow(robust), "\n")

permissive <- df_long[abs(delta_prob) >= 0.05]
cat("Permissive effects (|Δprob| >= 0.05):", nrow(permissive), "\n")

Variants with rsID   : 81780 
Variants without rsID: 0 
Variants matched to SuSiE CS table: 81780 


gene,clusterName,type,geneNameStr,geneClassStr,cell_type,sample_ontology_term,sample_ontology_type,rsid,CHR,⋯,susierss_SE,susierss_P,susierss_INFO,susierss_is_indep_snp,susierss_indep_snp_method,susierss_snp_gene,susierss_snp_func,susierss_NOVEL,susierss_b2_replicated,susierss_b2_replicated_bonf
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
"chr1:1910241..1910262,+",p1@CATG00000072713.1,promoter,CATG00000072713.1,lncRNA_antisense,CL:0000034,stem cell,cell_type,rs13302948,chr1,⋯,0.00609,4.17714e-08,0.997427,FALSE,NA,NA,NA,,,
"chr1:1910241..1910262,+",p1@CATG00000072713.1,promoter,CATG00000072713.1,lncRNA_antisense,CL:0000037,hematopoietic stem cell,cell_type,rs13302948,chr1,⋯,0.00609,4.17714e-08,0.997427,FALSE,NA,NA,NA,,,
"chr1:1910241..1910262,+",p1@CATG00000072713.1,promoter,CATG00000072713.1,lncRNA_antisense,CL:0000047,neuronal stem cell,cell_type,rs13302948,chr1,⋯,0.00609,4.17714e-08,0.997427,FALSE,NA,NA,NA,,,
"chr1:1910241..1910262,+",p1@CATG00000072713.1,promoter,CATG00000072713.1,lncRNA_antisense,CL:0000049,common myeloid progenitor,cell_type,rs13302948,chr1,⋯,0.00609,4.17714e-08,0.997427,FALSE,NA,NA,NA,,,
"chr1:1910241..1910262,+",p1@CATG00000072713.1,promoter,CATG00000072713.1,lncRNA_antisense,CL:0000056,myoblast,cell_type,rs13302948,chr1,⋯,0.00609,4.17714e-08,0.997427,FALSE,NA,NA,NA,,,
"chr1:1910241..1910262,+",p1@CATG00000072713.1,promoter,CATG00000072713.1,lncRNA_antisense,CL:0000057,fibroblast,cell_type,rs13302948,chr1,⋯,0.00609,4.17714e-08,0.997427,FALSE,NA,NA,NA,,,



Final columns:
 [1] "gene"                        "clusterName"                
 [3] "type"                        "geneNameStr"                
 [5] "geneClassStr"                "cell_type"                  
 [7] "sample_ontology_term"        "sample_ontology_type"       
 [9] "rsid"                        "CHR"                        
[11] "POS"                         "REF"                        
[13] "ALT"                         "dist"                       
[15] "strand"                      "delta_prob"                 
[17] "susierss_trait"              "susierss_risk_loci_num"     
[19] "susierss_cs"                 "susierss_pip"               
[21] "susierss_pip_rank"           "susierss_locus_gene"        
[23] "susierss_locus_start"        "susierss_locus_end"         
[25] "susierss_locus_cytoband"     "susierss_CHR"               
[27] "susierss_POS"                "susierss_EA"                
[29] "susierss_NEA"                "susierss_EAF"               
[31] "sus

In [ ]:
head(permissive)

gene,clusterName,type,geneNameStr,geneClassStr,cell_type,sample_ontology_term,sample_ontology_type,rsid,CHR,⋯,susierss_SE,susierss_P,susierss_INFO,susierss_is_indep_snp,susierss_indep_snp_method,susierss_snp_gene,susierss_snp_func,susierss_NOVEL,susierss_b2_replicated,susierss_b2_replicated_bonf
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
"chr1:2005462..2005537,+",p7@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0000353,parenchyma,tissue,rs13302948,chr1,⋯,0.006090,4.17714e-08,0.997427,FALSE,NA,NA,NA,,,
"chr1:2005046..2005122,+",p1@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0001896,medulla oblongata,tissue,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,
"chr1:2005046..2005122,+",p1@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0004875,nephrogenic cord,tissue,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,
"chr1:2005126..2005153,+",p5@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0004151,cardiac chamber,tissue,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,
"chr1:2005182..2005194,+",p8@PRKCZ,promoter,PRKCZ,coding_mRNA,UBERON:0003513,trunk blood vessel,tissue,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,
"chr1:2005228..2005248,+",p3@PRKCZ,promoter,PRKCZ,coding_mRNA,CL:0000134,mesenchymal stem cell,cell_type,rs2678942,chr1,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,


Same results as the quick run

# VCAN

> Read file with hg19 positions and rsID

In [ ]:
rsid_table <- fread("/network/iss/debette/users/bastien.rioux/mentr/input/mentr_input_hg19.csv")
head(rsid_table)

rsid,chr,pos,ea,nea
<chr>,<chr>,<int>,<chr>,<chr>
rs2459985,1,1999288,C,T
rs2803303,1,2001856,G,A
rs2678942,1,2004947,C,T
rs13302948,1,2008444,C,T
rs2678941,1,2001273,G,A
rs2254669,1,2006108,G,A


> Read CS

In [ ]:
cs_table <- read.xlsx("/network/iss/debette/users/bastien.rioux/mentr/input/susierss_cs_snps_20260417.xlsx")
cs_table <- cs_table %>% filter(grepl("VCAN", cs_table$locus_gene))
vcan_cs_rsid <- cs_table %>% pull(rsID) %>% unique()
length(vcan_cs_rsid)

[1] 32

> Format input file for VCAN locus

In [ ]:
rsid_table_vcan <- rsid_table %>% filter(rsid %in% vcan_cs_rsid) %>% select(-rsid)
rsid_table_vcan %>% write.table("/network/iss/debette/users/bastien.rioux/mentr/input/vcan_mentr_table.txt", 
                      sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)

# Load and clean chr-level chunk runs

Run section 'Load ontology mapping' first

This is the result from run_mentr_parallel.sh and submit_mentr_chunks.sh.

The script below lead per-chunk results (stored under mentr/output/chunk_*/qcd_res.txt.gz) into a single table with annotations and filtering.

In [ ]:
# Iterate through all_qcd_res files under /network/iss/debette/users/bastien.rioux/mentr/output
library(data.table)

# Define base directory
base_dir <- "/network/iss/debette/users/bastien.rioux/mentr/output"

# Find all all_qcd_res.txt.gz files recursively
files <- list.files(
  path       = base_dir,
  pattern    = "all_qcd_res\\.txt\\.gz$",
  recursive  = TRUE,
  full.names = TRUE
)

# Print files found (sanity check)
cat("Found", length(files), "files:\n")
print(files)

# Read and combine into a single data.table
all_qcd_res <- rbindlist(
  lapply(files, function(f) {
    cat("Reading:", f, "\n")
    fread(f)
  }),
  use.names = TRUE,
  fill      = TRUE   # handles any missing columns across chunks
)

cat("Final dimensions:", nrow(all_qcd_res), "rows x", ncol(all_qcd_res), "cols\n")

Found 8 files:
[1] "/network/iss/debette/users/bastien.rioux/mentr/output/chunk_1_chr1-2/output/all_qcd_res.txt.gz"  
[2] "/network/iss/debette/users/bastien.rioux/mentr/output/chunk_2_chr3-4/output/all_qcd_res.txt.gz"  
[3] "/network/iss/debette/users/bastien.rioux/mentr/output/chunk_3_chr5-6/output/all_qcd_res.txt.gz"  
[4] "/network/iss/debette/users/bastien.rioux/mentr/output/chunk_4_chr7-8/output/all_qcd_res.txt.gz"  
[5] "/network/iss/debette/users/bastien.rioux/mentr/output/chunk_5_chr9-10/output/all_qcd_res.txt.gz" 
[6] "/network/iss/debette/users/bastien.rioux/mentr/output/chunk_6_chr11-12/output/all_qcd_res.txt.gz"
[7] "/network/iss/debette/users/bastien.rioux/mentr/output/chunk_7_chr13-15/output/all_qcd_res.txt.gz"
[8] "/network/iss/debette/users/bastien.rioux/mentr/output/chunk_9_chr19-21/output/all_qcd_res.txt.gz"
Reading: /network/iss/debette/users/bastien.rioux/mentr/output/chunk_1_chr1-2/output/all_qcd_res.txt.gz 
Reading: /network/iss/debette/users/bastien.rioux/mentr/

In [ ]:
# Keep only reliable records
dim(all_qcd_res)
all_res_qced <- all_qcd_res[ref_matched == TRUE]
dim(all_res_qced)

[1] 216361    356

[1] 216361    356

In [ ]:
head(all_qcd_res)

CHR,POS,REF,ALT,dist,gene,strand,ref_matched,CL:0000034,CL:0000037,⋯,UBERON:0009722,UBERON:0009854,UBERON:0010133,UBERON:0010225,UBERON:0010260,UBERON:0010323,UBERON:0010743,UBERON:0011135,UBERON:0011137,LCL154
<chr>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<lgl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr1,15522936,G,A,44091,"chr1:15478839..15478845,-",-,TRUE,0,0,⋯,0,0,0,0,0,0,0,0,0,0
chr1,15558045,C,A,79200,"chr1:15478839..15478845,-",-,TRUE,0,0,⋯,0,0,0,0,0,0,0,0,0,0
chr1,15559672,G,A,80827,"chr1:15478839..15478845,-",-,TRUE,0,0,⋯,0,0,0,0,0,0,0,0,0,0
chr1,15560025,C,G,81180,"chr1:15478839..15478845,-",-,TRUE,0,0,⋯,0,0,0,0,0,0,0,0,0,0
chr1,15558250,A,T,79405,"chr1:15478839..15478845,-",-,TRUE,0,0,⋯,0,0,0,0,0,0,0,0,0,0
chr1,15558105,A,T,79260,"chr1:15478839..15478845,-",-,TRUE,0,0,⋯,0,0,0,0,0,0,0,0,0,0


> Read .csv file (hg19) containing rsid and chr/pos/ea/nea (needs to be downloaded directly from DNA Nexus with wget in mentr/input/)

DNA nexus path: /shivai_analyses/gwas/postgwas/mentr/input/

In CLI:
- cd /network/iss/debette/users/bastien.rioux/mentr/input/
- wget https://dl.ew2.dnanex.us/F/D/J20j9J68bJ4XPXG2x9vfXB1VfP9JXyv3YfXJjYv8/mentr_input_hg19.csv

In [ ]:
rsid_table <- fread("/network/iss/debette/users/bastien.rioux/mentr/input/mentr_input_hg19.csv")

# ── Harmonize CHR format ──────────────────────────────────────────────────────
rsid_table[, chr := paste0("chr", chr)]

cat("rsid_table chr examples: ", head(rsid_table$chr), "\n")
head(rsid_table)


rsid_table chr examples:  chr1 chr1 chr1 chr1 chr1 chr1 


rsid,chr,pos,ea,nea
<chr>,<chr>,<int>,<chr>,<chr>
rs2459985,chr1,1999288,C,T
rs2803303,chr1,2001856,G,A
rs2678942,chr1,2004947,C,T
rs13302948,chr1,2008444,C,T
rs2678941,chr1,2001273,G,A
rs2254669,chr1,2006108,G,A


> CS file to map rsID

DNA nexus path: /shivai_analyses/gwas/postgwas/susie_r/

in CLI:
- cd /network/iss/debette/users/bastien.rioux/mentr/input/ 
- wget https://dl.ew2.dnanex.us/F/D/JZvVvjQqj309jZjzFKzKK7YY4y4kby69Y21fz81X/susierss_cs_snps_20260417.xlsx

In [ ]:
cs_table <- read.xlsx("/network/iss/debette/users/bastien.rioux/mentr/input/susierss_cs_snps_20260417.xlsx")
# Add susierss prefix
rsid_col <- "rsID"   # ← change if column is named differently e.g. "rsid"
colnames(cs_table) <- ifelse(
  colnames(cs_table) == rsid_col,
  colnames(cs_table),                            # keep rsID as-is
  paste0("susierss_", colnames(cs_table))        # prefix all others
)
cat("\ncs_table columns after prefix:\n")
print(colnames(cs_table))


cs_table columns after prefix:
 [1] "susierss_trait"              "susierss_risk_loci_num"     
 [3] "susierss_cs"                 "rsID"                       
 [5] "susierss_pip"                "susierss_pip_rank"          
 [7] "susierss_locus_gene"         "susierss_locus_start"       
 [9] "susierss_locus_end"          "susierss_locus_cytoband"    
[11] "susierss_CHR"                "susierss_POS"               
[13] "susierss_EA"                 "susierss_NEA"               
[15] "susierss_EAF"                "susierss_BETA"              
[17] "susierss_SE"                 "susierss_P"                 
[19] "susierss_INFO"               "susierss_is_indep_snp"      
[21] "susierss_indep_snp_method"   "susierss_snp_gene"          
[23] "susierss_snp_func"           "susierss_NOVEL"             
[25] "susierss_b2_replicated"      "susierss_b2_replicated_bonf"


In [ ]:
cs_table

,susierss_trait,susierss_risk_loci_num,susierss_cs,rsID,susierss_pip,susierss_pip_rank,susierss_locus_gene,susierss_locus_start,susierss_locus_end,susierss_locus_cytoband,⋯,susierss_SE,susierss_P,susierss_INFO,susierss_is_indep_snp,susierss_indep_snp_method,susierss_snp_gene,susierss_snp_func,susierss_NOVEL,susierss_b2_replicated,susierss_b2_replicated_bonf
,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,bianca_deep_wmh,1,1,rs2459985,0.0858,1/39,PRKCZ,1067849,3067849,1p36.33,⋯,0.006052,1.97120e-08,0.994385,TRUE,Both,PRKCZ,intronic,Yes,No,No
2,bianca_deep_wmh,1,1,rs2803303,0.0663,2/39,PRKCZ,1067849,3067849,1p36.33,⋯,0.006053,2.63942e-08,0.995566,FALSE,NA,NA,NA,,,
3,bianca_deep_wmh,1,1,rs2678942,0.0624,3/39,PRKCZ,1067849,3067849,1p36.33,⋯,0.006023,2.80590e-08,0.996439,FALSE,NA,NA,NA,,,
4,bianca_deep_wmh,1,1,rs13302948,0.0447,4/39,PRKCZ,1067849,3067849,1p36.33,⋯,0.006090,4.17714e-08,0.997427,FALSE,NA,NA,NA,,,
5,bianca_deep_wmh,1,1,rs2678941,0.0387,5/39,PRKCZ,1067849,3067849,1p36.33,⋯,0.006051,4.78087e-08,0.996247,FALSE,NA,NA,NA,,,
6,bianca_deep_wmh,1,1,rs2254669,0.0341,6/39,PRKCZ,1067849,3067849,1p36.33,⋯,0.006021,5.46071e-08,0.996432,FALSE,NA,NA,NA,,,
7,bianca_deep_wmh,1,1,rs3121829,0.0321,7/39,PRKCZ,1067849,3067849,1p36.33,⋯,0.006080,6.02443e-08,0.998924,FALSE,NA,NA,NA,,,
8,bianca_deep_wmh,1,1,rs2490550,0.0314,8/39,PRKCZ,1067849,3067849,1p36.33,⋯,0.006077,6.17649e-08,0.997630,FALSE,NA,NA,NA,,,
9,bianca_deep_wmh,1,1,rs2490552,0.0301,9/39,PRKCZ,1067849,3067849,1p36.33,⋯,0.006078,6.47913e-08,0.997793,FALSE,NA,NA,NA,,,


> Format output

Susie rss values are merged into a single column (multiple rows for some rsID)

In [ ]:
library(data.table)

# ── Fixed columns (metadata) ──────────────────────────────────────────────────
meta_cols <- c("CHR", "POS", "REF", "ALT", "dist", "gene", "strand", "ref_matched")

# ── Cell type columns ─────────────────────────────────────────────────────────
cell_cols <- setdiff(colnames(all_res_qced), meta_cols)

# ── Melt to long format ───────────────────────────────────────────────────────
df_long <- melt(all_res_qced,
                id.vars       = meta_cols,
                variable.name = "cell_type",
                value.name    = "delta_prob")

# ── Join rsID ─────────────────────────────────────────────────────────────────
df_long <- merge(
  df_long,
  rsid_table[, .(rsid, chr, pos, ea, nea)],
  by.x  = c("CHR", "POS", "ALT", "REF"),
  by.y  = c("chr", "pos", "ea",  "nea"),
  all.x = TRUE
)

n_total   <- nrow(df_long)
n_matched <- df_long[!is.na(rsid), .N]
cat("Variants with rsID   :", n_matched, "\n")
cat("Variants without rsID:", n_total - n_matched, "\n")

# ── Join ontology ─────────────────────────────────────────────────────────────
df_long <- merge(
  df_long,
  ontology[, .(sample_ontology_ID,
               sample_ontology_term,
               sample_ontology_type)],
  by.x  = "cell_type",
  by.y  = "sample_ontology_ID",
  all.x = TRUE
)

# ── Join CAGE info ────────────────────────────────────────────────────────────
df_long <- merge(
  df_long,
  cage_info[, .(clusterID,
                clusterName,
                type,
                geneNameStr,
                geneClassStr)],
  by.x  = "gene",
  by.y  = "clusterID",
  all.x = TRUE
)

# ── Prepare SuSiE RSS CS table ────────────────────────────────────────────────
# Convert to data.table (read.xlsx returns a data.frame)
setDT(cs_table)

# Step 1: Build a human-readable summary string per row
cs_table[, trait_summary := paste0(
  susierss_trait,
  " (locus: ", susierss_risk_loci_num, " / ", susierss_locus_gene,
  ", CS: ",    susierss_cs,
  ", PIP = ",  round(susierss_pip, 3),
  ", rank = ", susierss_pip_rank, ")"
)]

# Step 2: Collapse multiple rows per rsID → one row with "|"-separated string
cs_collapsed <- cs_table[, .(
  trait = paste(trait_summary, collapse = " | ")
), by = rsID]

cat("Unique rsIDs in SuSiE CS table:", nrow(cs_collapsed), "\n")

# ── Join SuSiE RSS CS table by rsid (1-to-1, no cartesian risk) ──────────────
df_long <- merge(
  df_long,
  cs_collapsed,          # rsID + trait (collapsed summary string)
  by.x  = "rsid",
  by.y  = "rsID",
  all.x = TRUE           # keep all df_long rows
)

# Check match rate
n_in_cs <- df_long[!is.na(trait), .N]
cat("Variants matched to SuSiE CS table:", n_in_cs, "\n")
cat("Variants unmatched                :", df_long[is.na(trait), .N], "\n")

# ── Reorder columns ───────────────────────────────────────────────────────────
col_order <- c("gene", "clusterName", "type", "geneNameStr", "geneClassStr",
               "cell_type", "sample_ontology_term", "sample_ontology_type",
               "rsid", "CHR", "POS", "REF", "ALT", "dist", "strand",
               "delta_prob",
               "trait")   # ← single collapsed SuSiE column

df_long <- df_long[, ..col_order]

# Preview
head(df_long)
cat("\nFinal columns:\n")
print(colnames(df_long))

# ── Filter effects ────────────────────────────────────────────────────────────
robust <- df_long[abs(delta_prob) >= 0.1]
cat("Robust effects     (|Δprob| >= 0.1) :", nrow(robust), "\n")

permissive <- df_long[abs(delta_prob) >= 0.05]
cat("Permissive effects (|Δprob| >= 0.05):", nrow(permissive), "\n")

Variants with rsID   : 75293628 
Variants without rsID: 0 
Unique rsIDs in SuSiE CS table: 10396 
Variants matched to SuSiE CS table: 61618272 
Variants unmatched                : 13675356 


gene,clusterName,type,geneNameStr,geneClassStr,cell_type,sample_ontology_term,sample_ontology_type,rsid,CHR,POS,REF,ALT,dist,strand,delta_prob,trait
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<chr>,<dbl>,<chr>
chr4:4854782-4854915,e1@ADDG04004854782.E,enhancer,ADDG04004854782.E,enhancer_locus,CL:0000034,stem cell,cell_type,rs10000804,chr4,4864299,G,A,9467,N,0,"shiva_deep_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 3, PIP = 0.1, rank = 4/12) | shiva_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 2, PIP = 0.054, rank = 8/54)"
chr4:4854782-4854915,e1@ADDG04004854782.E,enhancer,ADDG04004854782.E,enhancer_locus,CL:0000037,hematopoietic stem cell,cell_type,rs10000804,chr4,4864299,G,A,9467,N,0,"shiva_deep_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 3, PIP = 0.1, rank = 4/12) | shiva_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 2, PIP = 0.054, rank = 8/54)"
chr4:4854782-4854915,e1@ADDG04004854782.E,enhancer,ADDG04004854782.E,enhancer_locus,CL:0000047,neuronal stem cell,cell_type,rs10000804,chr4,4864299,G,A,9467,N,0,"shiva_deep_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 3, PIP = 0.1, rank = 4/12) | shiva_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 2, PIP = 0.054, rank = 8/54)"
chr4:4854782-4854915,e1@ADDG04004854782.E,enhancer,ADDG04004854782.E,enhancer_locus,CL:0000049,common myeloid progenitor,cell_type,rs10000804,chr4,4864299,G,A,9467,N,0,"shiva_deep_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 3, PIP = 0.1, rank = 4/12) | shiva_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 2, PIP = 0.054, rank = 8/54)"
chr4:4854782-4854915,e1@ADDG04004854782.E,enhancer,ADDG04004854782.E,enhancer_locus,CL:0000056,myoblast,cell_type,rs10000804,chr4,4864299,G,A,9467,N,0,"shiva_deep_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 3, PIP = 0.1, rank = 4/12) | shiva_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 2, PIP = 0.054, rank = 8/54)"
chr4:4854782-4854915,e1@ADDG04004854782.E,enhancer,ADDG04004854782.E,enhancer_locus,CL:0000057,fibroblast,cell_type,rs10000804,chr4,4864299,G,A,9467,N,0,"shiva_deep_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 3, PIP = 0.1, rank = 4/12) | shiva_pvs (locus: 16 / STX18-AS1/MSX1,LOC101928306, CS: 2, PIP = 0.054, rank = 8/54)"



Final columns:
 [1] "gene"                 "clusterName"          "type"                
 [4] "geneNameStr"          "geneClassStr"         "cell_type"           
 [7] "sample_ontology_term" "sample_ontology_type" "rsid"                
[10] "CHR"                  "POS"                  "REF"                 
[13] "ALT"                  "dist"                 "strand"              
[16] "delta_prob"           "trait"               
Robust effects     (|Δprob| >= 0.1) : 826 
Permissive effects (|Δprob| >= 0.05): 16646 


In [ ]:
# Write
write.xlsx(permissive, 
           "/network/iss/debette/users/bastien.rioux/mentr/output_formatted/mentr_permissive.xlsx")

In [ ]:
robust

gene,clusterName,type,geneNameStr,geneClassStr,cell_type,sample_ontology_term,sample_ontology_type,rsid,CHR,POS,REF,ALT,dist,strand,delta_prob,trait
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<chr>,<dbl>,<chr>
chr8:49426952-49427328,e1@ADDG08049426952.E,enhancer,ADDG08049426952.E,enhancer_locus,CL:0000084,T cell,cell_type,rs10092171,chr8,49427167,G,A,-5,N,-0.1045710,"shiva_wmh (locus: 30 / UBE2V2,LOC101929268, CS: 1, PIP = 0.035, rank = 3/76)"
chr8:49426952-49427328,e1@ADDG08049426952.E,enhancer,ADDG08049426952.E,enhancer_locus,CL:0000542,lymphocyte,cell_type,rs10092171,chr8,49427167,G,A,-5,N,-0.1055892,"shiva_wmh (locus: 30 / UBE2V2,LOC101929268, CS: 1, PIP = 0.035, rank = 3/76)"
chr8:49426952-49427328,e1@ADDG08049426952.E,enhancer,ADDG08049426952.E,enhancer_locus,CL:0000576,monocyte,cell_type,rs10092171,chr8,49427167,G,A,-5,N,-0.1135946,"shiva_wmh (locus: 30 / UBE2V2,LOC101929268, CS: 1, PIP = 0.035, rank = 3/76)"
chr8:49426952-49427328,e1@ADDG08049426952.E,enhancer,ADDG08049426952.E,enhancer_locus,CL:0000798,gamma-delta T cell,cell_type,rs10092171,chr8,49427167,G,A,-5,N,-0.1108362,"shiva_wmh (locus: 30 / UBE2V2,LOC101929268, CS: 1, PIP = 0.035, rank = 3/76)"
chr8:49426952-49427328,e1@ADDG08049426952.E,enhancer,ADDG08049426952.E,enhancer_locus,CL:0000842,mononuclear cell,cell_type,rs10092171,chr8,49427167,G,A,-5,N,-0.2121058,"shiva_wmh (locus: 30 / UBE2V2,LOC101929268, CS: 1, PIP = 0.035, rank = 3/76)"
chr8:49426952-49427328,e1@ADDG08049426952.E,enhancer,ADDG08049426952.E,enhancer_locus,CL:0000860,classical monocyte,cell_type,rs10092171,chr8,49427167,G,A,-5,N,-0.1057864,"shiva_wmh (locus: 30 / UBE2V2,LOC101929268, CS: 1, PIP = 0.035, rank = 3/76)"
chr8:49426952-49427328,e1@ADDG08049426952.E,enhancer,ADDG08049426952.E,enhancer_locus,CL:0002582,visceral preadipocyte,cell_type,rs10092171,chr8,49427167,G,A,-5,N,-0.1217962,"shiva_wmh (locus: 30 / UBE2V2,LOC101929268, CS: 1, PIP = 0.035, rank = 3/76)"
chr8:49426952-49427328,e1@ADDG08049426952.E,enhancer,ADDG08049426952.E,enhancer_locus,LCL154,NA,NA,rs10092171,chr8,49427167,G,A,-5,N,-0.1072644,"shiva_wmh (locus: 30 / UBE2V2,LOC101929268, CS: 1, PIP = 0.035, rank = 3/76)"
chr8:49427546-49427722,e1@ADDG08049427546.E,enhancer,ADDG08049427546.E,enhancer_locus,UBERON:0001829,major salivary gland,tissue,rs10092171,chr8,49427167,G,A,-466,N,-0.1135367,"shiva_wmh (locus: 30 / UBE2V2,LOC101929268, CS: 1, PIP = 0.035, rank = 3/76)"
